# 01 · LangGraph: an explicit graph with built-in memory

LangGraph describes an agent as a **graph**: a typed state, nodes that are plain functions
reading and updating that state, and edges that say what runs next. We build one graph per
stage and draw it each time, so you can watch it grow: one node, then a triage step with four
paths, then a review loop with memory attached.

Three stages:
1. a basic bot
2. triage and scenarios
3. memory, a reviewer and tracing

In [ ]:
# Reload the src modules automatically when they change (no kernel restart needed).
%load_ext autoreload
%autoreload 2

In [ ]:
import warnings
from typing import Annotated, TypedDict

import yaml
from dotenv import load_dotenv

from agentic_demo import config
from agentic_demo.models import Answer, Review, Triage
from agentic_demo.paths import EXAMPLES_PATH, MEMORY_DIR, QUESTIONS_PATH, SERVER_PATH
from agentic_demo.prompts import load_prompt

load_dotenv()
warnings.filterwarnings("ignore", message=".*langchain.mcp.*")
MODEL = f"anthropic:{config.ANTHROPIC_MODEL}"  # or f"openai:{config.OPENAI_MODEL}"
questions = {q["scenario"]: q for q in yaml.safe_load(QUESTIONS_PATH.read_text())}

Shared parts come from `src/agentic_demo`: the output models, the prompt loader and the paths.
The model is one string, `provider:name`; the same line works for OpenAI, Anthropic and Ollama.
`langchain.mcp` is in beta, hence the warning filter.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware
from langchain.mcp import MCPAdapter
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages

from agentic_demo.display import show_mermaid

async with MCPAdapter(SERVER_PATH) as sales_db:
    tools = await sales_db.list_tools()
print([tool.name for tool in tools])

call_limit = ModelCallLimitMiddleware(run_limit=15)  # default limit is about 10,000

In [ ]:
import json

from langchain_core.utils.function_calling import convert_to_openai_tool

for tool in tools:
    print(f"{tool.name}: {tool.description}")
print(json.dumps(convert_to_openai_tool(tools[1]), indent=2))  # what the model receives

`MCPAdapter` starts our MCP server over stdio and turns its two tools into LangChain tools;
`create_agent` builds the model-tools loop that we will put inside a node; `StateGraph` is the
graph we draw with `show_mermaid`.

**How the model knows the tools.** The server declares each tool from a plain Python function:
its name, its docstring and its type-hinted parameters become the JSON schema printed above.
`MCPAdapter.list_tools()` fetches those declarations, and `create_agent` sends them to the model
with every request as function definitions. The model never runs anything itself: when it wants
data it answers with a *tool call*, a small JSON object such as
`{"name": "run_sql", "arguments": {"query": "SELECT ..."}}`. LangGraph's tool node executes that
call over MCP and appends the result to the conversation as a tool message, and the model
continues with that message in its context. So the docstring in `mcp_server/server.py` is prompt
text: it is all the model knows about what the tool does.

In [ ]:
SCENARIOS = ["answer", "clarify", "out_of_scope", "not_allowed"]
DEFAULT_REPLY = {
    "out_of_scope": "I only know the sales table: units, gross and net revenue.",
    "not_allowed": "I can only read the sales data, not change it.",
}


async def ask(graph, question: str, thread: str = "default") -> None:
    print(f"Q: {question}")
    run_config = {"configurable": {"thread_id": thread}}
    await graph.ainvoke({"messages": [("user", question)]}, run_config)

`ask` is the whole chat UI: one `ainvoke` on a graph, with a `thread_id` that names the
conversation; it prints the question first, so every output reads as a dialogue. The two fixed replies are what the bot says for questions it must not or cannot
answer, from stage 2 on.

In [ ]:
def say(scenario: str, text: str) -> dict:
    print(f"[{scenario}] {text}")
    return {"messages": [("assistant", text)]}


def clarify(state) -> dict:
    return say("clarify", state["triage"].question)


def decline(state) -> dict:
    return say(state["triage"].scenario, DEFAULT_REPLY[state["triage"].scenario])


SIMPLE_NODES = {"clarify": clarify, "out_of_scope": decline, "not_allowed": decline}

Three scenario nodes need no model at all: `clarify` asks the question that the triage step
wrote, `decline` prints a fixed reply. Both append their text to the conversation, so a
follow-up reads naturally. Only the `answer` path runs an agent, and that path differs per stage.

## Stage 1: a basic bot in a graph of one node

A `StateGraph` has three parts. The **state** is a typed dict; here it holds the conversation
`messages` and the typed `answer`. **Nodes** are plain Python functions: each receives the
state and returns only the keys it changes. **Edges** connect the nodes, from the special
`START` to the special `END`. LangGraph runs the nodes in that order and merges each result
into the state; `add_messages` tells it to append to the message list instead of replacing it.

Here the whole agent sits inside a single node: a minimal prompt, the two MCP tools and a typed
`Answer`. The graph around it gives no support, so every decision is left to the model.

**How nodes connect.** Not by passing return values hand to hand: nodes talk through the state.
A node returns a dict with some of the state's keys, LangGraph merges it into the state, and the
next node reads whatever keys it needs. The `State` class is the contract that makes nodes
compatible: a node may only write keys declared there, a node that reads a key relies on an
earlier node having written it, and a key that should accumulate rather than be overwritten,
like `messages`, carries a reducer (`add_messages` appends, and replaces a message with the same
id). The edges decide only the order.

In [ ]:
bot = create_agent(
    MODEL,
    tools,
    system_prompt=load_prompt("basic"),
    response_format=Answer,
    middleware=[call_limit],
)


class BotState(TypedDict):
    messages: Annotated[list, add_messages]
    answer: Answer

The bot is the stage 1 agent: minimal prompt, both tools, typed answer. `BotState` is what the
graph carries between nodes: the conversation and that answer.

In [ ]:
async def bot_node(state: BotState) -> dict:
    out = await bot.ainvoke({"messages": state["messages"]})
    answer = out["structured_response"]
    print(f"A: {answer.answer}\n  SQL: {answer.sql}")
    return {"messages": out["messages"], "answer": answer}


builder = StateGraph(BotState)
builder.add_node("bot", bot_node)
builder.add_edge(START, "bot")
builder.add_edge("bot", END)
graph = builder.compile()
show_mermaid(graph.get_graph().draw_mermaid())

The node calls the agent with the messages from the state and hands back the new messages and
the typed answer. Three boxes, two edges: the agent's own model-tools loop is inside `bot`
(`graph.get_graph(xray=True)` would expose it).

In [ ]:
for item in questions.values():
    await ask(graph, item["question"])

The second question says "there": with no memory of the previous answer the bot cannot know
it means the Americas, so it guesses or sums every region. Also watch the revenue itself
(gross or net, picked silently), the margin question (no costs in the table) and the DELETE:
only the MCP server stops it. The graph has no checkpointer, so every question starts from an
empty state.

In [ ]:
result = await graph.ainvoke({"messages": [("user", questions["answer"]["question"])]})
for message in result["messages"]:
    calls = [f"{c['name']}({c['args']})" for c in getattr(message, "tool_calls", [])]
    text = " ".join(message.text.split())[:100]
    print(f"{message.type:>5} | {calls or text}")

**Where the SQL runs.** The whole model-tools loop happened inside the one node; reading the
messages back shows it. The model first calls `get_schema` and gets the table definition as a
tool message. It then calls `run_sql` with the SQL it wrote; the MCP server process runs that
query on the read-only SQLite file and returns at most 50 rows, which arrive as another tool
message. The model reads the rows from that message and writes the answer as the last AI message,
as JSON text or as a final tool call depending on the provider, which `create_agent` parses
into the `Answer` object. No Python code of ours ever handles the rows: they exist only in that
tool message, which is why the prompt says to take numbers only from `run_sql`.

## Stage 2: triage first, then the right path

Real assistants do not hand every message to the tool-using agent. The first node, **triage**,
is one structured model call without tools: it reads the conversation and returns a `Triage`
with the scenario and, for `clarify`, the question to ask. A **conditional edge**, a plain
function that looks at the state and returns the name of the next node (drawn as dashed lines),
then sends the run down one of four paths. Only `answer` runs the analyst agent with the
database tools; `clarify` asks back; `out_of_scope` and `not_allowed` print a fixed reply, so a
prohibited request never touches the tools.

A **checkpointer** saves the state after every step, per `thread_id`. The next call with the
same thread continues the conversation, which is how the user's reply to a clarifying question
becomes an `answer`; a new thread starts clean.

In [ ]:
triage = create_agent(
    MODEL, system_prompt=load_prompt("triage"), response_format=Triage
)
analyst = create_agent(
    MODEL,
    tools,
    system_prompt=load_prompt("analyst"),
    response_format=Answer,
    middleware=[call_limit],
)

Two agents from the same factory. `triage` has no tools, so it is a single model call that
returns a `Triage`; `analyst` has the two MCP tools and returns an `Answer`, like the stage 1
bot but with the analyst prompt.

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]
    triage: Triage


async def triage_node(state: State) -> dict:
    out = await triage.ainvoke({"messages": state["messages"]})
    return {"triage": out["structured_response"]}


async def answer_node(state: State) -> dict:
    out = await analyst.ainvoke({"messages": state["messages"]})
    answer = out["structured_response"]
    print(f"[answer] {answer.answer}\n  SQL: {answer.sql}")
    return {"messages": out["messages"]}

`triage_node` stores the decision in the state; `answer_node` runs the analyst on the
conversation, prints the result and keeps the new messages. Both call their agent like a
function.

In [ ]:
builder = StateGraph(State)
builder.add_node("triage", triage_node)
builder.add_node("answer", answer_node)
for name, node in SIMPLE_NODES.items():
    builder.add_node(name, node)
for scenario in SCENARIOS:
    builder.add_edge(scenario, END)
builder.add_edge(START, "triage")
builder.add_conditional_edges(
    "triage", lambda state: state["triage"].scenario, SCENARIOS
)
graph = builder.compile(checkpointer=InMemorySaver())
show_mermaid(graph.get_graph().draw_mermaid())

The dashed lines are the conditional edge: the lambda returns the scenario, which is also the
name of the target node. The checkpointer is given at compile time; swap `InMemorySaver` for
`SqliteSaver` to keep threads across restarts.

In [ ]:
for scenario, item in questions.items():
    await ask(graph, item["question"], thread="demo")
    if scenario == "clarify":
        await ask(graph, item["reply"], thread="demo")

print("--- a new thread knows nothing")
await ask(graph, questions["clarify"]["again"], thread="demo-2")

All four questions run in one thread, like one user session: "there" is resolved from the
previous answer, and when triage asks back, the user's reply in the same thread turns the
question into an `answer`. A new thread starts empty, so even the self-contained version of
the question is asked back. That gap is what stage 3 closes.

Ask your own question here; reuse the `thread` to continue a conversation.

In [ ]:
# await ask(graph, "Which product sold the most units in Q1 2026?", thread="live")

## Stage 3: memory, a reviewer and tracing

### Long-term memory: the store

The checkpointer remembers one thread. A **store** is memory outside any thread: documents
under a namespace, each with a key and a value, kept here in a SQLite file, so they survive a
new conversation and a kernel restart. With an **embedding index**, `search(..., query=...)`
returns the entries closest in meaning to a question; that is how the analyst finds the two
most relevant verified examples. The glossary, the company's own definitions, lives in the
same store.

A **dynamic prompt** is middleware: a function that rebuilds the system prompt on every model
call, here from the store. The agents do not change; what they read does. Triage reads it too,
so a term the company has defined no longer needs a clarifying question.

In [ ]:
import shutil

shutil.rmtree(
    MEMORY_DIR / "langgraph", ignore_errors=True
)  # skip this cell to keep the memory

Every rehearsal starts clean. Skip this cell to show that memory survives a kernel restart.

In [ ]:
import sqlite3

from langchain.embeddings import init_embeddings
from langgraph.store.sqlite import SqliteStore

memory_dir = MEMORY_DIR / "langgraph"
memory_dir.mkdir(parents=True, exist_ok=True)
connection = sqlite3.connect(
    memory_dir / "memory.db", isolation_level=None, check_same_thread=False
)
embed = init_embeddings("openai:text-embedding-3-small")
store = SqliteStore(
    connection, index={"dims": 1536, "embed": embed, "fields": ["question"]}
)
store.setup()

One SQLite file under `data/memory/langgraph/`. The index embeds the `question` field of every
stored document with an OpenAI embedding model, so this cell needs `OPENAI_API_KEY` even when
the chat model is Anthropic.

In [ ]:
from langchain.tools import ToolRuntime

for number, example in enumerate(yaml.safe_load(EXAMPLES_PATH.read_text())):
    store.put(("examples",), f"example-{number}", example)
hits = store.search(("examples",), query="units per product", limit=2)
print([hit.value["question"] for hit in hits])


def remember(term: str, definition: str, runtime: ToolRuntime) -> str:
    """Save a company definition, for example revenue = net revenue."""
    runtime.store.put(("glossary",), term, {"definition": definition}, index=False)
    return f"Saved: {term} = {definition}"

Two ways to write memory: the five verified examples are loaded once, at setup time, and the
semantic search already finds the closest ones; the glossary is written at run time by the
`remember` tool when the user teaches a definition. `runtime` is injected by LangGraph and
carries the store.

In [ ]:
from langchain.agents.middleware import ModelRequest, dynamic_prompt


def memory_prompt(name: str):
    @dynamic_prompt
    def prompt(request: ModelRequest) -> str:
        question = [m.content for m in request.messages if m.type == "human"][-1]
        hits = request.runtime.store.search(("examples",), query=question, limit=2)
        notes = ["Follow these definitions, do not ask; save new ones with remember:"]
        notes += [f"Example: {hit.value}" for hit in hits]
        for entry in request.runtime.store.search(("glossary",)):
            notes.append(f"Definition: {entry.key} = {entry.value['definition']}")
        return load_prompt(name) + "\n\n" + "\n".join(notes)

    return prompt

`memory_prompt` returns middleware that composes the system prompt on every model call: the
prompt file, the two examples closest to the latest user message, and every glossary entry.

In [ ]:
triage = create_agent(
    MODEL, response_format=Triage, middleware=[memory_prompt("triage")]
)
analyst = create_agent(
    MODEL,
    [*tools, remember],
    response_format=Answer,
    middleware=[call_limit, memory_prompt("analyst")],
)
reviewer = create_agent(
    MODEL, response_format=Review, middleware=[memory_prompt("reviewer")]
)

All three agents read memory through the dynamic prompt: triage learns which terms are defined,
the analyst gets the examples and the definitions, the reviewer the definitions. Only the
analyst has tools, and only it can write to the glossary.

### A reviewer in the same graph

A second agent checks each answer before it is shown. Triage still goes first; the `answer`
path now runs the analyst, then the reviewer, and the reviewer's conditional edge decides what
happens next. The loop is the dashed edge back to the analyst in the diagram.

**What the reviewer sees.** One message with every user turn of the thread (the question may
depend on earlier turns, like "there" or the "net revenue" reply), the SQL the analyst reported
and its answer sentence, plus the glossary and the examples through the dynamic prompt. It has
no tools and does not get the rows: it judges the query as text. By then the SQL has already
been executed by the analyst, so the review checks a real answer; a retry writes and runs a new
query.

**Two rounds at most.** First attempt, first review. If rejected, the reason is appended to the
conversation as a user message and the analyst tries once more, which is reviewed again. After
that second review the `answer` node prints the result whatever the verdict, with both verdicts
printed above it: nothing can loop forever, and a disagreement is shown rather than hidden.

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]
    triage: Triage
    answer: Answer
    review: Review | None
    retried: bool


def after_triage(state: State) -> str:
    scenario = state["triage"].scenario
    return "analyst" if scenario == "answer" else scenario


def after_reviewer(state: State) -> str:
    return "answer" if state["review"].approved or state["retried"] else "analyst"

Three more state keys and two routing functions. `after_triage` sends answerable questions to
the analyst and the other scenarios straight to their node; `after_reviewer` closes the loop
after one retry.

In [ ]:
async def triage_node(state: State) -> dict:
    out = await triage.ainvoke({"messages": state["messages"]})
    return {"triage": out["structured_response"]}


async def analyst_node(state: State) -> dict:
    review = state.get("review")
    out = await analyst.ainvoke({"messages": state["messages"]})
    again = review is not None and not review.approved
    update = {"messages": out["messages"], "answer": out["structured_response"]}
    return update | {"review": None, "retried": again}

The triage node is the stage 2 one with the memory-aware agent. The analyst node clears the
pending review and records whether this pass was the retry (a rejected review was waiting). `update | {...}` is Python's dict union: one new dict from both, the right side winning on duplicate keys.

In [ ]:
async def reviewer_node(state: State) -> dict:
    answer = state["answer"]
    turns = "\n".join(m.content for m in state["messages"] if m.type == "human")
    text = f"Conversation:\n{turns}\nSQL: {answer.sql}\nAnswer: {answer.answer}"
    out = await reviewer.ainvoke({"messages": [("user", text)]})
    review = out["structured_response"]
    print(f"[review] approved={review.approved}: {review.reason}")
    note = [("user", f"Rejected: {review.reason}. Fix the SQL.")]
    return {"review": review, "messages": [] if review.approved else note}


def show_answer(state: State) -> None:
    answer = state["answer"]
    print(f"[answer] {answer.answer}\n  SQL: {answer.sql}")

The reviewer gets the user's turns, the SQL and the answer as text, prints its verdict and, on
rejection, adds the reason to the conversation so the retry sees it. `show_answer` is the
`answer` node now: it only prints what the loop approved.

In [ ]:
builder = StateGraph(State)
builder.add_node("triage", triage_node)
builder.add_node("analyst", analyst_node)
builder.add_node("reviewer", reviewer_node)
builder.add_node("answer", show_answer)
for name, node in SIMPLE_NODES.items():
    builder.add_node(name, node)
for scenario in SCENARIOS:
    builder.add_edge(scenario, END)
builder.add_edge(START, "triage")
builder.add_edge("analyst", "reviewer")
builder.add_conditional_edges("triage", after_triage, ["analyst", *SCENARIOS[1:]])
builder.add_conditional_edges("reviewer", after_reviewer, ["answer", "analyst"])
graph = builder.compile(checkpointer=InMemorySaver(), store=store)
show_mermaid(graph.get_graph().draw_mermaid())

Compared with stage 2: the `answer` path became `analyst -> reviewer -> answer`, with the dashed
edge back to `analyst` that makes the loop. The graph is compiled with the store, because the
nested agents inherit the runtime of the graph that calls them.

In [ ]:
await ask(graph, questions["answer"]["question"], thread="teach")
await ask(graph, questions["clarify"]["question"], thread="teach")
await ask(graph, questions["clarify"]["reply"], thread="teach")
print("--- a new thread, with memory")
await ask(graph, questions["clarify"]["again"], thread="later")
print(
    "glossary:", [(i.key, i.value["definition"]) for i in store.search(("glossary",))]
)

The user teaches the bot once and `remember` writes the glossary. In the new thread, triage
reads the definition and sends the self-contained question straight to the analyst, which
answers with net revenue, and the reviewer checks that choice against the same glossary.

### Tracing with LangSmith

Every model call, tool call and graph step of a run can be recorded as a **trace**: inputs,
outputs, latency and tokens, nested the way the graph ran them. LangChain sends traces to
LangSmith when two environment variables are set, `LANGSMITH_TRACING=true` and
`LANGSMITH_API_KEY`; nothing in the code changes. `LANGSMITH_PROJECT` names the project the
runs land in. This cell fetches the latest run and prints a link to open during the talk.

In [ ]:
import os

from langchain_core.tracers.langchain import wait_for_all_tracers
from langsmith import Client
from langsmith.utils import tracing_is_enabled

if tracing_is_enabled():
    wait_for_all_tracers()  # the tracer uploads in the background
    client = Client()
    project = os.environ.get("LANGSMITH_PROJECT", "default")
    run = next(client.list_runs(project_name=project, is_root=True, limit=1))
    print(f"{run.name}: {run.status}, {run.total_tokens} tokens")
    print(client.get_run_url(run=run))
else:
    print("Set LANGSMITH_TRACING=true and LANGSMITH_API_KEY in .env first.")

The latest root run is the graph call just above; open the link to walk through triage, the
analyst with its tool calls and the reviewer, with the time and tokens of each step.

In [ ]:
await ask(
    graph, "What was our revenue per region in the last 3 months?", thread="live-3"
)

Ask your own question: the examples and the glossary shape the SQL, and the reviewer checks it.

### The same graph on a local model

One line changes the model. `qwen3:8b` on Ollama is slower and less reliable with structured
output, which is exactly the trade-off to show.

In [ ]:
import time

MODEL = f"ollama:{config.LOCAL_MODEL}"  # or ChatOllama(model=config.LOCAL_MODEL, num_ctx=16384)
analyst = create_agent(
    MODEL,
    [*tools, remember],
    response_format=Answer,
    middleware=[call_limit, memory_prompt("analyst")],
)
start = time.perf_counter()
await ask(graph, questions["clarify"]["again"], thread="local")
print(f"{time.perf_counter() - start:.1f} s with {MODEL}")

The graph is unchanged: `analyst_node` picks up the rebuilt `analyst`, so the same loop now runs
the local model inside that node while triage and the reviewer stay on the cloud model. This
cell is not run by the automated check (it needs Ollama); run it live.

## Wrap-up

- **Easy:** the control flow is an explicit, drawable graph: triage first, tools only on the
  `answer` path, a reviewer loop in a few lines of edges.
- **Easy:** memory is built in: a checkpointer per thread, a store with semantic search across
  threads, read by every agent through one dynamic prompt.
- **Hard:** more concepts before the first answer: state and reducers, threads, middleware,
  the runtime, and the structured-response key in the output dict.